# Simple Pad — Q3D Capacitance

Public synthetic tutorial candidate `orpen-simple-pad-public-tutorial`
under `scgsim-docs-1.0.2-20261006`: semantic state CONVERGING, delivery
LOCAL CANDIDATE, no test writes. Owner: OrPen SC PDK. Canonical layout
authority is `square_pad_capacitor`; reusable runtime belongs to SCGSim.
The authored pad and connected surrounding Ground are finite Al (0–0.2
um), above one 500 um Si die (-500–0 um). There is no second die or
fabricated inter-die gap. The source locator sheet spans
x=99.999–120.001, y=-1–1 um: 20.002 by 2 um, centered at (110, 0),
directed +X. It overlaps each terminal by one source DBU (0.001 um) for
exact contact binding; physical metal and the 20 um gap are unchanged.
The component owns this overlap metadata.

Source receipt: OrPen base `36b6e6b`; SCGSim runtime 1.0.1 stable source
`9c032e398f2ebabc2dc53965b2bde5c346a061bc` (develop `8361bd902` has
unchanged runtime). Use that exact runtime cohort; local source
authoring is not an installed-version or native-model observation.
Quarto execution is disabled. This checkpoint has no native geometry,
mesh, solve, capacitance, frequency, EPR or adequacy claim. Run IDs
below identify new preparations, not pre-existing evidence; change the
ID for each new run. Native search/convergence controls are inputs, not
research adequacy Gates.

This branch requests CG-only extraction at 5 GHz: no AC R/L and no
junction L/C branch. Q3D’s surrounding Region is open, not assigned to a
grounded Region net. The pad is Signal and the single connected ring is
Ground. The PDK superconducting Al record is lowered by SCGSim to native
PEC; finite geometry is preserved, not a finite-conductivity loss
measurement. Palace’s natural electrostatic exterior and Q3D’s open
Region are not claimed equivalent.

## Setup and Imports

In [ ]:
from pathlib import Path
import os
import subprocess
import importlib.metadata

import gdsfactory as gf
from IPython.display import display
from scgsim.aedt import (
    AedtResources, LayerImport, ObjectBinding, PdkMaterial,
    Q3dNetSpec, Q3dSpec, MatrixRunControl, prepare_handoff,
    resolve_results,
)

import orpen_sc_pdk
from orpen_sc_pdk import get_material_records, get_interface_preset_records, validate_interface_preset_records
from orpen_sc_pdk.tech import LAYER, get_single_die_layer_stack

## Setup and Run Controls

In [ ]:
WORKFLOW_ACTION = "prepare_handoff"  # prepare_handoff | run | analyze_handoff
RUN_ID = "simple_pad_aedt_q3d_20261006_02"
OUTPUT_ROOT = Path.cwd() / ".artifacts" / "SimplePad"
RUN_DIR = OUTPUT_ROOT / RUN_ID
PAD_SIDE_UM = 200.0
GAP_UM = 20.0
DIE_SIDE_UM = 1000.0
JUNCTION_WIDTH_UM = 2.0
VACUUM_PADDING_UM = (250.0, 250.0, 1000.0)
AEDT_VERSION = "2024.2"  # Current SCGSim Eigenmode facade binds this runtime version.
PYAEDT_VERSION = "1.3.0"
SCGSIM_RUNTIME_REVISION = "9c032e398f2ebabc2dc53965b2bde5c346a061bc"
SETUP_NAME = "Setup1"
MAXIMUM_PASSES = 10
CONVERGENCE_PERCENT = 1.0
RESOURCES = AedtResources(cores=4, ram_limit_percent=80)
EXTRACTION_FREQUENCY_GHZ = 5.0
SOLVE_AC_RL = False
GROUNDED_REGION_NET = None
SOURCE_GDS = OUTPUT_ROOT / "source_gds" / f"{RUN_ID}.gds"
PAD_IMPORT_LAYER = (300, 0)
GROUND_IMPORT_LAYER = (301, 0)
SUBSTRATE_IMPORT_LAYER = (302, 0)
if WORKFLOW_ACTION not in {"prepare_handoff", "run", "analyze_handoff"}:
    raise ValueError("Unknown WORKFLOW_ACTION")

## Create Simulation Component / Coupon

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    orpen_sc_pdk.activate()
    gf.clear_cache()
    component = gf.get_component(
        "square_pad_capacitor", pad_side_um=PAD_SIDE_UM, gap_um=GAP_UM,
        die_side_um=DIE_SIDE_UM, junction_width_um=JUNCTION_WIDTH_UM,
    )
    layer_stack = get_single_die_layer_stack()
    material_records = get_material_records()

    display(component)

## Initialize AEDT Project / App

Preparation creates a portable SCGSim package, not a live Desktop
session. Only the generated runner in the explicit `run` action owns
native AEDT.

In [ ]:
display({
    "requested_aedt": AEDT_VERSION, "requested_pyaedt": PYAEDT_VERSION,
    "installed_pyaedt": importlib.metadata.version("pyaedt"),
    "installed_scgsim": importlib.metadata.version("scgsim"),
    "runtime_source": SCGSIM_RUNTIME_REVISION, "run_id": RUN_ID,
})

## Import GDS and Build the HFSS/Q3D/Q2D Model

Q3D has no GeometryPlan facade. Split the canonical **positive** DRAW
using GDSFactory boolean operations against the same parameter-bound pad
footprint. This selects the central pad and the remaining connected
Ground without DBU conversion or a second fabrication model. Dedicated
positive import layers make native object binding explicit. Substrate
extent and z/thickness come from the canonical component and selected
PDK stack; the locator sheet and negative ground mask are not imported
as metal.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    pad_selector = gf.components.rectangle(
        size=(PAD_SIDE_UM, PAD_SIDE_UM), centered=True, layer=LAYER.D0_TOP_M1_DRAW,
    )
    pad_geometry = gf.boolean(
        A=component, B=pad_selector, operation="and", layer=PAD_IMPORT_LAYER,
        layer1=LAYER.D0_TOP_M1_DRAW, layer2=LAYER.D0_TOP_M1_DRAW,
    )
    ground_geometry = gf.boolean(
        A=component, B=pad_selector, operation="not", layer=GROUND_IMPORT_LAYER,
        layer1=LAYER.D0_TOP_M1_DRAW, layer2=LAYER.D0_TOP_M1_DRAW,
    )
    substrate_geometry = component.extract(layers=[LAYER.D0_SUBSTRATE_AREA])
    substrate_geometry.remap_layers({LAYER.D0_SUBSTRATE_AREA: SUBSTRATE_IMPORT_LAYER})
    q3d_geometry = gf.Component()
    q3d_geometry << pad_geometry
    q3d_geometry << ground_geometry
    q3d_geometry << substrate_geometry
    SOURCE_GDS.parent.mkdir(parents=True, exist_ok=True)
    # Exclusive creation keeps a repeated run ID from silently replacing source GDS.
    with SOURCE_GDS.open("xb") as stream:
        q3d_geometry.write_gds(stream.name, with_metadata=False)
    metal_level = layer_stack.layers["D0_TOP_M1"]
    substrate_level = layer_stack.layers["D0_SUBSTRATE"]
    layer_imports = (
        LayerImport(300, 0, "PAD", metal_level.zmin, metal_level.zmin + metal_level.thickness),
        LayerImport(301, 0, "GROUND", metal_level.zmin, metal_level.zmin + metal_level.thickness),
        LayerImport(302, 0, "SUBSTRATE", substrate_level.zmin,
                    substrate_level.zmin + substrate_level.thickness),
    )
    object_bindings = (
        ObjectBinding("PAD_1", 300, "signal", "Al"),
        ObjectBinding("GROUND_2", 301, "ground", "Al"),
        ObjectBinding("SUBSTRATE_3", 302, "substrate", "Si"),
    )

## Geometry Verification

This is source inspection, not native geometry readback. The actual
runner must retain native material, object, net, bounds and assignment
observations in its receipt; analyze those observations after retrieval.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display({"component_bbox_um": str(component.dbbox()), "ports": component.ports})
    display(q3d_geometry)
    display({"layer_imports": layer_imports, "object_bindings": object_bindings})

## Materials and Boundaries

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    materials = {
        material_id: PdkMaterial(
            material_id, material_records[material_id]["material_kind"],
            material_records[material_id]["is_superconducting"],
            material_records[material_id]["aedt_library_name"],
        )
        for material_id in ("Al", "Si", "vacuum")
    }

## Ports / Nets / Excitations

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    nets = (
        Q3dNetSpec("pad", "Signal", ("PAD_1",), "PAD_1", "-Z", "PAD_1", "+Z"),
        Q3dNetSpec("ground", "Ground", ("GROUND_2",)),
    )

## Simulation Setup

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    run_control = MatrixRunControl(
        setup_name=SETUP_NAME, frequency_ghz=EXTRACTION_FREQUENCY_GHZ,
        maximum_passes=MAXIMUM_PASSES, convergence_percent=CONVERGENCE_PERCENT,
    )

## Simulation Configuration

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    px, py, pz = VACUUM_PADDING_UM
    spec = Q3dSpec(
        gds_path=SOURCE_GDS, project_name=RUN_ID, design_name="SimplePadQ3d",
        materials=materials, vacuum_material_id="vacuum",
        layer_imports=layer_imports, object_bindings=object_bindings, nets=nets,
        run_control=run_control, region_padding_um=(px, px, py, py, pz, pz),
        solve_ac_rl=SOLVE_AC_RL, grounded_region_net=GROUNDED_REGION_NET,
        aedt_version=AEDT_VERSION, pyaedt_version=PYAEDT_VERSION,
    )
    display(spec)

## Solve and Export

A fresh handoff directory is required by SCGSim. Preparation never opens
AEDT or solves. Change the action to `run` only for explicitly
authorized native host execution; that branch invokes only the generated
runner.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    HANDOFF = prepare_handoff(spec=spec, output_dir=RUN_DIR, resources=RESOURCES)
    display(HANDOFF)
    if WORKFLOW_ACTION == "run":
        subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Adaptive-Pass Convergence / Solver Diagnostics

Use a fresh kernel and an explicit returned run directory, not live
preparation state. The strict resolver reads sealed receipt identity,
copied inputs and native readback. Missing or corrupt artifacts fail;
nothing synthesizes a successful return. Retain the exact
preparation-side spec/manifest identity independently when transporting
the package.

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    RETURNED_RUN_DIR = Path(os.environ["SIMPLE_PAD_RETURNED_RUN_DIR"])
    result = resolve_results(RETURNED_RUN_DIR)
    # Trust/identity and native numerical/performance evidence precede physics.
    result.show_all_results(show_details=True)

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display(result.physics_results())

### Simulation Performance / Benchmarks

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display(result.simulation_benchmark())

## Save and Release AEDT

The generated runner saves the exact project and releases only its owned
Desktop session. The strict resolver requires successful save/release
and hash-bound output. No native outcome exists at this source
checkpoint.

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display({"project": result.project_path, "receipt": result.receipt_path})
elif WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display(HANDOFF.archive_path)